# Clase 8 · Tarea 02 — Mini proyecto: generador de reportes de datos

### Proyecto integrador final · pipeline completo

En esta tarea construyes, pieza a pieza, un **generador automático de reportes**
que puede analizar cualquier DataFrame y producir un resumen ejecutivo completo.

```
DataFrame
    │
    ▼
[Parte 1] perfilar_dataset      → dimensiones, nulos, tipos
    │
    ▼
[Parte 2] estadisticas_numericas → media, std, skewness por columna
    │
    ▼
[Parte 3] detectar_outliers      → filas atípicas con z-score
    │
    ▼
[Parte 4] top_valores            → ranking de grupos
    │
    ▼
[Parte 5] correlacion_variables  → relaciones entre columnas
    │
    ▼
[Parte 6] generar_reporte        → INTEGRACIÓN TOTAL
```

**Instrucciones**

1. Implementa cada parte en su celda correspondiente.
2. Las partes 3–6 dependen de las anteriores — resuélvelas en orden.
3. La Parte 6 integra todo: si las 5 anteriores pasan, la 6 también pasará.

> 🎯 **Meta:** al terminar, `generar_reporte(df)` funciona sobre
> **cualquier** DataFrame CSV con columnas numéricas.

In [ ]:
# Identifícate (esto ayuda si entregas el notebook).
NOMBRE = ""   # ✏️ escribe tu nombre completo
print("Tarea lista para resolver. Ejecuta cada bloque de tests tras implementar.")

## Ejercicio 1 · Parte 1 — perfilar_dataset(df)

Implementa `perfilar_dataset(df)` que reciba un DataFrame y devuelva
un dict con el perfil completo:

```python
{
  'filas': int,
  'columnas': int,
  'duplicados': int,
  'nulos_total': int,
  'nulos_por_columna': dict,   # solo col con nulos
  'columnas_numericas': list,
  'columnas_categoricas': list,
}
```

**Pista:** usa `df.select_dtypes(include='number').columns.tolist()` y
`df.select_dtypes(exclude='number').columns.tolist()`.

In [ ]:
import pandas as pd
import numpy as np

def perfilar_dataset(df):
    nulos = df.isnull().sum()
    nulos_por_col = {c: int(v) for c, v in nulos[nulos > 0].items()}
    return {
        'filas':                len(df),
        'columnas':             len(df.columns),
        'duplicados':           int(df.duplicated().sum()),
        'nulos_total':          int(df.isnull().sum().sum()),
        'nulos_por_columna':    nulos_por_col,
        'columnas_numericas':   df.select_dtypes(include='number').columns.tolist(),
        'columnas_categoricas': df.select_dtypes(exclude='number').columns.tolist(),
    }

In [ ]:
# === Tests visibles · Ejercicio 1 ===
import pandas as pd, numpy as np
_df_p = pd.DataFrame({'a': [1.0, np.nan, 1.0], 'b': ['x', 'y', 'x']})
_r1 = perfilar_dataset(_df_p)
assert _r1['filas'] == 3
assert _r1['columnas'] == 2
assert _r1['nulos_total'] == 1
print("✅ Ejercicio 1: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 1 ===
assert _r1['duplicados'] == 1
assert _r1['nulos_por_columna'] == {'a': 1}
assert 'a' in _r1['columnas_numericas']
assert 'b' in _r1['columnas_categoricas']
print("✅ Ejercicio 1: tests adicionales superados.")

## Ejercicio 2 · Parte 2 — estadisticas_numericas(df)

Implementa `estadisticas_numericas(df)` que devuelva un **DataFrame**
con una fila por columna numérica y las columnas:
`media`, `mediana`, `std`, `minimo`, `maximo`, `skewness`.

Redondea todos los valores a 2 decimales.

**Pista:** filtra con `df.select_dtypes(include='number')` y
construye el DataFrame a mano con `pd.DataFrame({...})`.

In [ ]:
import pandas as pd
import numpy as np

def estadisticas_numericas(df):
    num = df.select_dtypes(include='number')
    resultado = pd.DataFrame({
        'media':    num.mean(),
        'mediana':  num.median(),
        'std':      num.std(),
        'minimo':   num.min(),
        'maximo':   num.max(),
        'skewness': num.skew(),
    }).round(2)
    return resultado

In [ ]:
# === Tests visibles · Ejercicio 2 ===
import pandas as pd
_df_e = pd.DataFrame({'x': [1.0, 2.0, 3.0], 'y': [10.0, 20.0, 30.0], 'z': ['a','b','c']})
_r2 = estadisticas_numericas(_df_e)
assert isinstance(_r2, pd.DataFrame)
assert 'media' in _r2.columns and 'skewness' in _r2.columns
assert set(_r2.index) == {'x', 'y'}
print("✅ Ejercicio 2: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 2 ===
assert abs(_r2.loc['x', 'media'] - 2.0) < 1e-3
assert abs(_r2.loc['y', 'mediana'] - 20.0) < 1e-3
_df_e2 = pd.DataFrame({'a': [1.0, 1.0, 1.0]})
_r2b = estadisticas_numericas(_df_e2)
assert _r2b.loc['a', 'minimo'] == _r2b.loc['a', 'maximo']
print("✅ Ejercicio 2: tests adicionales superados.")

## Ejercicio 3 · Parte 3 — detectar_outliers(df, columna, umbral)

Implementa `detectar_outliers(df, columna, umbral=3.0)` que:

1. Calcule el z-score de `columna` usando NumPy.
2. Devuelva el subconjunto de filas con `|z| > umbral`.
3. Incluya una columna extra `'zscore'` con el valor del z.

**Ejemplo:**
```python
out = detectar_outliers(df, 'monto')
# out tiene todas las columnas de df + 'zscore'
```

In [ ]:
import pandas as pd
import numpy as np

def detectar_outliers(df, columna, umbral=3.0):
    valores = df[columna].to_numpy().astype(float)
    z = (valores - valores.mean()) / valores.std()
    mask = np.abs(z) > umbral
    resultado = df[mask].copy()
    resultado['zscore'] = z[mask]
    return resultado.reset_index(drop=True)

In [ ]:
# === Tests visibles · Ejercicio 3 ===
import pandas as pd, numpy as np
_s_big = pd.Series(list(range(1, 21)) + [500])
_df_o = pd.DataFrame({'val': _s_big})
_out3 = detectar_outliers(_df_o, 'val')
assert isinstance(_out3, pd.DataFrame)
assert 'zscore' in _out3.columns
assert 500 in _out3['val'].values
print("✅ Ejercicio 3: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 3 ===
_out3b = detectar_outliers(_df_o, 'val', umbral=10.0)
assert len(_out3b) == 0
_out3c = detectar_outliers(_df_o, 'val', umbral=1.0)
assert len(_out3c) >= len(_out3)
print("✅ Ejercicio 3: tests adicionales superados.")

## Ejercicio 4 · Parte 4 — top_valores(df, columna_grupo, columna_valor, n)

Implementa `top_valores(df, columna_grupo, columna_valor, n=5)` que:

1. Agrupe por `columna_grupo`.
2. Sume `columna_valor` por grupo.
3. Devuelva los `n` grupos con mayor suma, como DataFrame con:
   - índice = `columna_grupo`
   - columna `'total'`
   - columna `'pct'` (porcentaje del total general, redondeado a 1 decimal)

**Ejemplo:**
```python
top_valores(df, 'ciudad', 'monto', 3)
# las 3 ciudades con mayor monto + su porcentaje del total
```

In [ ]:
import pandas as pd

def top_valores(df, columna_grupo, columna_valor, n=5):
    totales = df.groupby(columna_grupo)[columna_valor].sum()
    top     = totales.nlargest(n)
    gran_total = totales.sum()
    resultado = pd.DataFrame({
        'total': top,
        'pct':   (top / gran_total * 100).round(1),
    })
    return resultado

In [ ]:
# === Tests visibles · Ejercicio 4 ===
import pandas as pd
_df_t = pd.DataFrame({'grp': list('AAABBBCCC'), 'val': [10,20,30,5,5,5,1,1,1]})
_top4 = top_valores(_df_t, 'grp', 'val', 2)
assert isinstance(_top4, pd.DataFrame)
assert 'total' in _top4.columns and 'pct' in _top4.columns
assert len(_top4) == 2
print("✅ Ejercicio 4: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 4 ===
assert _top4.index[0] == 'A'
assert abs(_top4.loc['A', 'total'] - 60) < 1e-3
assert abs(_top4['pct'].sum() - (60+15)/78*100) < 0.2
print("✅ Ejercicio 4: tests adicionales superados.")

## Ejercicio 5 · Parte 5 — correlacion_variables(df)

Implementa `correlacion_variables(df)` que calcule la **matriz de
correlación** de las columnas numéricas y devuelva un DataFrame
redondeado a 3 decimales.

También devuelve el **par de variables más correlacionado** (excluyendo
la diagonal) como una tupla `(var1, var2, r)`.

Devuelve: `(matriz_corr, (var1, var2, r))`

**Pista:** usa `df.corr()`, luego convierte a numpy con `.to_numpy().copy()`,
pon el diagonal en 0 con `np.fill_diagonal`, y usa `np.argmax` + `np.unravel_index`.

In [ ]:
import pandas as pd
import numpy as np

def correlacion_variables(df):
    num  = df.select_dtypes(include='number')
    corr = num.corr().round(3)
    arr  = corr.to_numpy().copy().astype(float)
    np.fill_diagonal(arr, 0)
    idx  = np.unravel_index(np.argmax(np.abs(arr)), arr.shape)
    var1 = corr.index[idx[0]]
    var2 = corr.columns[idx[1]]
    r    = float(corr.iloc[idx[0], idx[1]])
    return corr, (var1, var2, r)

In [ ]:
# === Tests visibles · Ejercicio 5 ===
import pandas as pd, numpy as np
_df_c = pd.DataFrame({'x': [1.0,2.0,3.0,4.0], 'y': [2.0,4.0,6.0,8.0], 'z': [1.0,0.0,1.0,0.0]})
_mat, _par = correlacion_variables(_df_c)
assert isinstance(_mat, pd.DataFrame)
assert isinstance(_par, tuple) and len(_par) == 3
print("✅ Ejercicio 5: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 5 ===
assert abs(_mat.loc['x', 'y'] - 1.0) < 1e-3
assert set(_par[:2]) == {'x', 'y'}
assert abs(_par[2]) >= abs(_mat.loc['x', 'z'])
print("✅ Ejercicio 5: tests adicionales superados.")

## Ejercicio 6 · Parte 6 — generar_reporte(df, titulo)

Implementa `generar_reporte(df, titulo='Reporte de datos')` que
integre todas las partes anteriores y devuelva un dict con el
reporte completo:

```python
{
  'titulo':             str,
  'perfil':             dict,          # perfilar_dataset
  'estadisticas':       pd.DataFrame,  # estadisticas_numericas
  'n_outliers':         int,            # count de detectar_outliers (umbral=3)
  'correlacion_max':    tuple,          # (var1, var2, r) de correlacion_variables
}
```

Usa `'monto'` como columna para `detectar_outliers`. Si el df no tiene
`'monto'`, usa la primera columna numérica del perfil.

In [ ]:
import pandas as pd
import numpy as np

def generar_reporte(df, titulo='Reporte de datos'):
    perfil = perfilar_dataset(df)
    estadisticas = estadisticas_numericas(df)
    col_num = 'monto' if 'monto' in perfil['columnas_numericas'] else perfil['columnas_numericas'][0]
    out = detectar_outliers(df, col_num)
    _, par_corr = correlacion_variables(df)
    return {
        'titulo':          titulo,
        'perfil':          perfil,
        'estadisticas':    estadisticas,
        'n_outliers':      len(out),
        'correlacion_max': par_corr,
    }

In [ ]:
# === Tests visibles · Ejercicio 6 ===
import pandas as pd, numpy as np
_df_r = pd.DataFrame({'monto': list(range(1,21)) + [500], 'cat': ['a']*21})
_rep = generar_reporte(_df_r, titulo='Test')
assert isinstance(_rep, dict)
assert {'titulo','perfil','estadisticas','n_outliers','correlacion_max'} <= set(_rep.keys())
assert _rep['titulo'] == 'Test'
print("✅ Ejercicio 6: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 6 ===
assert _rep['n_outliers'] >= 1
assert isinstance(_rep['estadisticas'], pd.DataFrame)
assert isinstance(_rep['correlacion_max'], tuple)
assert _rep['perfil']['filas'] == 21
print("✅ Ejercicio 6: tests adicionales superados.")

---
## ¡Proyecto final completado!

Construiste un generador de reportes que combina todos los conceptos del curso:

| Parte | Concepto |
|---|---|
| `perfilar_dataset` | Estructuras de datos + Pandas |
| `estadisticas_numericas` | `groupby`, `agg`, `skew` |
| `detectar_outliers` | NumPy vectorizado |
| `top_valores` | `nlargest`, porcentajes |
| `correlacion_variables` | Álgebra lineal con NumPy |
| `generar_reporte` | **Integración total** |

### Reto opcional (sin calificar)

- Agrega una función `visualizar_reporte(reporte)` que genere 2–3 gráficos
  a partir del dict devuelto por `generar_reporte`.
- Convierte el reporte a un string formateado legible para imprimir en
  consola (similar a un dashboard en texto).
- ¿Cómo exportarías el reporte a un archivo JSON para compartirlo?
  (**Pista:** `json.dumps`, pero los DataFrames necesitan `.to_dict()`.)

### Reflexión final

> *"El objetivo del análisis de datos no es producir gráficos o tablas —*
> *es tomar mejores decisiones con evidencia."*

Las herramientas que aprendiste (Python, NumPy, Pandas) son medios, no fines.
Lo importante es el pensamiento algorítmico que desarrollaste: descomponer un
problema complejo en funciones pequeñas, verificables y reutilizables.

Eso es exactamente lo que hacen los científicos de datos profesionales.

> ✅ **Curso completado.** Tienes las bases para seguir con machine learning,
> visualización avanzada, SQL, o cualquier herramienta del ecosistema de datos.